# Neural Networks and Forward Propagation

A neural network is a composition of simple functions — layers of weighted sums followed by nonlinear activations. **Forward propagation** is the process of passing an input through every layer to produce an output. It is the inference step: given trained weights and a new input, compute the prediction.

Understanding forward propagation mathematically — in matrix form, layer by layer — is essential before learning backpropagation, building networks from scratch, or reading framework code. This notebook develops the full mathematical machinery and implements forward propagation in NumPy from the ground up.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Network Architecture and Notation

A feedforward neural network consists of layers of neurons connected in one direction — from input to output, with no cycles.

```
  Layer 0        Layer 1         Layer 2         Layer 3
  (input)       (hidden)        (hidden)        (output)

   x₁ ─────┐    h₁ ─────┐      h₁' ─────┐       ŷ₁
           ├─→         ├─→            ├─→
   x₂ ─────┤    h₂ ─────┤      h₂' ─────┤       ŷ₂
           │            │                 │
   x₃ ─────┘    h₃ ─────┘      h₃' ─────┘
```

**Notation used throughout:**

| Symbol | Meaning |
|--------|--------|
| $L$ | Number of layers (excluding input) |
| $n^{[l]}$ | Number of neurons in layer $l$ |
| $\mathbf{a}^{[l]}$ | Activations (outputs) of layer $l$ |
| $\mathbf{z}^{[l]}$ | Pre-activation (weighted sum before activation) |
| $\mathbf{W}^{[l]}$ | Weight matrix for layer $l$, shape $(n^{[l]}, n^{[l-1]})$ |
| $\mathbf{b}^{[l]}$ | Bias vector for layer $l$, shape $(n^{[l]}, 1)$ |
| $f^{[l]}$ | Activation function for layer $l$ |
| $m$ | Number of training examples (batch size) |

Layer 0 is the input: $\mathbf{a}^{[0]} = \mathbf{x}$. Each subsequent layer computes $\mathbf{z}^{[l]}$ then $\mathbf{a}^{[l]} = f^{[l]}(\mathbf{z}^{[l]})$.

---

## 2. A Single Neuron

The atomic unit of a neural network. For input vector $\mathbf{x} \in \mathbb{R}^n$:

$$z = \mathbf{w}^T \mathbf{x} + b = \sum_{i=1}^{n} w_i x_i + b$$

$$a = f(z)$$

where $\mathbf{w}$ is the weight vector, $b$ is the scalar bias, and $f$ is the activation function.

A neuron with sigmoid activation and binary cross-entropy loss is equivalent to **logistic regression**. A neuron with identity activation and MSE loss is equivalent to **linear regression**. The neuron is the universal building block.

In [ ]:
# Single neuron forward pass
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([1.0, 2.0, -1.0])       # 3 inputs
w = np.array([0.5, -0.3, 0.8])       # 3 weights
b = 0.1

z = np.dot(w, x) + b
a = sigmoid(z)

print(f"Input x:  {x}")
print(f"Weights w: {w}")
print(f"Bias b:   {b}")
print(f"z = w·x + b = {z:.4f}")
print(f"a = σ(z)  = {a:.4f}")

---

## 3. A Single Layer — Vectorized Form

A layer with $n^{[l]}$ neurons receiving input from $n^{[l-1]}$ neurons is computed in one matrix operation:

$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$

$$\mathbf{a}^{[l]} = f^{[l]}(\mathbf{z}^{[l]})$$

**Dimensions:**
- $\mathbf{W}^{[l]}$: $(n^{[l]} \times n^{[l-1]})$ — row $i$ contains weights for neuron $i$
- $\mathbf{b}^{[l]}$: $(n^{[l]} \times 1)$
- $\mathbf{a}^{[l-1]}$: $(n^{[l-1]} \times 1)$
- $\mathbf{z}^{[l]}$, $\mathbf{a}^{[l]}$: $(n^{[l]} \times 1)$

Each row of $\mathbf{W}^{[l]}$ corresponds to one neuron's weights. The matrix multiply computes all neurons in the layer simultaneously — this vectorization is what makes neural networks computationally efficient on GPUs.

In [ ]:
def relu(z):
    return np.maximum(0, z)

# Layer with 3 inputs → 4 neurons
a_prev = np.array([[1.0], [2.0], [-1.0]])   # (3, 1)
W = np.array([[0.1, 0.2, 0.3],
              [0.4, 0.5, 0.6],
              [0.7, 0.8, 0.9],
              [1.0, 1.1, 1.2]])              # (4, 3)
b = np.array([[0.1], [0.2], [0.3], [0.4]])   # (4, 1)

z = W @ a_prev + b
a = relu(z)

print(f"Input shape:      {a_prev.shape}")
print(f"Weight shape:     {W.shape}")
print(f"z (pre-activation):\n{z.T}")
print(f"a (post-activation):\n{a.T}")

---

## 4. Multi-Layer Forward Propagation

Forward propagation chains layer computations. For a network with $L$ layers, starting with $\mathbf{a}^{[0]} = \mathbf{x}$:

For $l = 1, 2, \ldots, L$:

$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$

$$\mathbf{a}^{[l]} = f^{[l]}(\mathbf{z}^{[l]})$$

The final output is $\hat{\mathbf{y}} = \mathbf{a}^{[L]}$.

**Example: 3-input → 4-hidden (ReLU) → 2-hidden (ReLU) → 1-output (sigmoid)**

This is a binary classifier. The hidden layers learn intermediate representations; the output neuron produces a probability.

In [ ]:
def forward_pass(x, weights, biases, activations):
    """
    Forward propagation through L layers.
    x:           input vector (n0, 1)
    weights:     list of W matrices [W1, W2, ..., WL]
    biases:      list of b vectors [b1, b2, ..., bL]
    activations: list of activation functions [f1, f2, ..., fL]
    Returns: cache of all z and a values, and final output.
    """
    cache = {"a": [x], "z": []}
    a = x
    for W, b, f in zip(weights, biases, activations):
        z = W @ a + b
        a = f(z)
        cache["z"].append(z)
        cache["a"].append(a)
    return cache

# Network: 3 → 4 → 2 → 1
np.random.seed(0)
W1 = np.random.randn(4, 3) * 0.5
b1 = np.zeros((4, 1))
W2 = np.random.randn(2, 4) * 0.5
b2 = np.zeros((2, 1))
W3 = np.random.randn(1, 2) * 0.5
b3 = np.zeros((1, 1))

x = np.array([[1.5], [0.8], [-0.3]])
cache = forward_pass(x, [W1, W2, W3], [b1, b2, b3], [relu, relu, sigmoid])

print("Layer-by-layer forward pass:")
for l in range(3):
    print(f"  Layer {l+1}: z shape {cache['z'][l].shape}, a shape {cache['a'][l+1].shape}")
print(f"\nFinal output ŷ = {cache['a'][-1][0, 0]:.4f}")

---

## 5. Batch Processing

In practice, we process multiple examples simultaneously as a **batch** for computational efficiency.

Instead of a single column vector $\mathbf{a}^{[l-1]}$ of shape $(n^{[l-1]} \times 1)$, we have a matrix of shape $(n^{[l-1]} \times m)$ where $m$ is the batch size — each **column** is one example.

$$\mathbf{Z}^{[l]} = \mathbf{W}^{[l]} \mathbf{A}^{[l-1]} + \mathbf{b}^{[l]}$$

The bias $\mathbf{b}^{[l]}$ is broadcast across all $m$ columns.

**Dimension cheat sheet for batch forward pass:**

| Tensor | Shape |
|--------|-------|
| $\mathbf{A}^{[0]} = \mathbf{X}$ | $(n^{[0]} \times m)$ |
| $\mathbf{W}^{[l]}$ | $(n^{[l]} \times n^{[l-1]})$ |
| $\mathbf{Z}^{[l]}$, $\mathbf{A}^{[l]}$ | $(n^{[l]} \times m)$ |
| $\mathbf{b}^{[l]}$ | $(n^{[l]} \times 1)$ — broadcast to $(n^{[l]} \times m)$ |

In [ ]:
# Batch forward pass: 3 features, batch of 5 examples
m = 5
X = np.random.randn(3, m)   # (3 features, 5 examples)

Z1 = W1 @ X + b1             # (4, 5) — bias broadcast across columns
A1 = relu(Z1)
Z2 = W2 @ A1 + b2
A2 = relu(Z2)
Z3 = W3 @ A2 + b3
Y_hat = sigmoid(Z3)          # (1, 5) — one prediction per example

print(f"Input X:      {X.shape}  (features × batch)")
print(f"Hidden A1:    {A1.shape}")
print(f"Hidden A2:    {A2.shape}")
print(f"Output Ŷ:     {Y_hat.shape}")
print(f"\nPredictions for 5 examples: {Y_hat.round(3).flatten()}")

---

## 6. Output Layers for Different Tasks

The output layer activation depends on the task:

| Task | Output neurons | Activation | Output interpretation |
|------|---------------|------------|----------------------|
| Binary classification | 1 | Sigmoid | Probability of class 1 |
| Multi-class classification | $K$ (one per class) | Softmax | Probability distribution over $K$ classes |
| Regression | 1 (or more) | Identity (linear) | Predicted numeric value |
| Multi-label classification | $K$ | Sigmoid (per neuron) | Independent probability per label |

### Softmax

For $K$ classes, softmax converts raw scores (logits) into a probability distribution:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

Properties: all outputs are in $(0, 1)$ and sum to 1. The class with the highest logit gets the highest probability.

In [ ]:
def softmax(z):
    """Numerically stable softmax."""
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# 3-class logits: shape (3 classes, 4 examples)
logits = np.array([[2.0, 0.5, 1.0, 0.1],
                    [1.0, 2.5, 1.0, 0.2],
                    [0.1, 0.3, 3.0, 0.3]])

probs = softmax(logits)   # softmax across classes (axis=0)

print("Logits (classes × examples):")
print(np.round(logits, 2))
print("\nSoftmax probabilities:")
print(np.round(probs, 3))
print(f"\nColumn sums (each example): {probs.sum(axis=0)}")
print(f"Predicted classes: {np.argmax(probs, axis=0)}")

---

## 7. Counting Parameters

The number of **trainable parameters** in a layer:

$$\text{params}^{[l]} = n^{[l]} \times n^{[l-1]} + n^{[l]} = n^{[l]} (n^{[l-1]} + 1)$$

Weights: $n^{[l]} \times n^{[l-1]}$. Biases: $n^{[l]}$.

Total parameters = sum across all layers. Parameter count determines model capacity, memory usage, and training time.

In [ ]:
def count_parameters(layer_sizes):
    """layer_sizes: list like [3, 4, 2, 1] for input + hidden + output."""
    total = 0
    print(f"{'Layer':>10s} {'Weights':>10s} {'Biases':>8s} {'Total':>8s}")
    print("-" * 40)
    for l in range(1, len(layer_sizes)):
        n_prev, n_curr = layer_sizes[l-1], layer_sizes[l]
        w = n_curr * n_prev
        b = n_curr
        total += w + b
        print(f"{n_prev}→{n_curr}:   {w:10d} {b:8d} {w+b:8d}")
    print(f"\nTotal parameters: {total}")
    return total

count_parameters([3, 4, 2, 1])
print()
count_parameters([784, 128, 64, 10])  # MNIST-like network

---

## 8. Building a Complete Network Class

We now assemble everything into a reusable neural network class with forward propagation, supporting arbitrary layer sizes and activation functions.

In [ ]:
class NeuralNetwork:
    """Fully-connected feedforward neural network."""

    def __init__(self, layer_sizes, activations):
        """
        layer_sizes: [n_input, n_hidden1, ..., n_output]
        activations: list of functions, one per layer (excluding input)
        """
        self.layer_sizes = layer_sizes
        self.activations = activations
        self.weights = []
        self.biases = []

        for l in range(1, len(layer_sizes)):
            W = np.random.randn(layer_sizes[l], layer_sizes[l-1]) * np.sqrt(2.0 / layer_sizes[l-1])
            b = np.zeros((layer_sizes[l], 1))
            self.weights.append(W)
            self.biases.append(b)

    def forward(self, X):
        """X: (n_features, m) matrix. Returns output and cache."""
        cache = {"a": [X], "z": []}
        a = X
        for W, b, f in zip(self.weights, self.biases, self.activations):
            z = W @ a + b
            a = f(z)
            cache["z"].append(z)
            cache["a"].append(a)
        return a, cache

    def predict(self, X):
        output, _ = self.forward(X)
        if output.shape[0] == 1:
            return (output > 0.5).astype(int)
        return np.argmax(output, axis=0)

    def count_params(self):
        return sum(W.size + b.size for W, b in zip(self.weights, self.biases))


# Test on moons dataset
X, y = make_moons(n_samples=300, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_t = scaler.fit_transform(X_tr).T   # (2, m) — features as rows
X_te_t = scaler.transform(X_te).T

nn = NeuralNetwork([2, 16, 8, 1], [relu, relu, sigmoid])
print(f"Network parameters: {nn.count_params()}")
output, cache = nn.forward(X_tr_t)
print(f"Output shape: {output.shape}  (1 output × {output.shape[1]} examples)")
print(f"Untrained accuracy: {accuracy_score(y_tr, nn.predict(X_tr_t).flatten()):.3f}")

---

## 9. Weight Initialization

Random weights must be initialized carefully. Poor initialization causes vanishing or exploding activations.

**Zero initialization** — all weights = 0. Every neuron in a layer learns the same thing. Useless.

**Random small values** — weights from $\mathcal{N}(0, 0.01)$. Activations shrink in deep networks (vanishing).

**Xavier initialization** — for sigmoid/tanh: $W \sim \mathcal{N}(0, \frac{1}{n^{[l-1]}})$. Keeps variance stable across layers.

**He initialization** — for ReLU: $W \sim \mathcal{N}(0, \frac{2}{n^{[l-1]}})$. Accounts for ReLU killing half the activations.

Our network class uses He initialization: `np.sqrt(2.0 / n_prev)` scaling.

In [ ]:
# Effect of initialization on activation magnitudes
n_in, n_hidden, n_layers, m = 100, 50, 10, 200
X_init = np.random.randn(n_in, m)

def propagate_with_init(init_scale):
    a = X_init
    for _ in range(n_layers):
        W = np.random.randn(n_hidden, a.shape[0]) * init_scale
        a = relu(W @ a)
    return np.std(a)

scales = {"Too small (0.01)": 0.01, "Xavier (1/√n)": 1/np.sqrt(n_in),
          "He ( √(2/n) )": np.sqrt(2.0/n_in), "Too large (1.0)": 1.0}

print(f"Activation std after {n_layers} layers:")
for name, scale in scales.items():
    std = propagate_with_init(scale)
    print(f"  {name:25s}: std = {std:.4f}")

---

## 10. The Computational Graph

Forward propagation can be viewed as a **computational graph** — a directed acyclic graph where nodes are operations (matrix multiply, add, activation) and edges carry tensors.

```
  x ──→ [W1· + b1] ──→ [ReLU] ──→ [W2· + b2] ──→ [ReLU] ──→ [W3· + b3] ──→ [σ] ──→ ŷ
```

Each node stores its output for use in backpropagation. The **cache** returned by our forward pass serves this purpose — it saves all intermediate $\mathbf{z}^{[l]}$ and $\mathbf{a}^{[l]}$ values needed to compute gradients.

Modern frameworks (PyTorch, TensorFlow) build this graph automatically. PyTorch builds it dynamically (eager mode); TensorFlow can compile it for optimization. Understanding the graph conceptually makes framework code transparent.

---

## 11. Forward Propagation as Function Composition

A neural network is a nested composition of functions:

$$\hat{\mathbf{y}} = f^{[L]} \circ \phi^{[L]} \circ f^{[L-1]} \circ \phi^{[L-1]} \circ \cdots \circ f^{[1]} \circ \phi^{[1]}(\mathbf{x})$$

where $\phi^{[l]}(\mathbf{a}) = \mathbf{W}^{[l]} \mathbf{a} + \mathbf{b}^{[l]}$ is the affine transformation and $f^{[l]}$ is the activation.

Each composition level transforms the input into a more abstract representation. The network as a whole is a single function $\hat{\mathbf{y}} = f_\theta(\mathbf{x})$ parameterized by all weights and biases $\theta = \{\mathbf{W}^{[l]}, \mathbf{b}^{[l]}\}_{l=1}^{L}$.

Training finds $\theta$ that minimizes loss. Forward propagation evaluates $f_\theta(\mathbf{x})$ for given $\theta$ and $\mathbf{x}$. Backpropagation computes $\frac{\partial L}{\partial \theta}$.

---

## 12. Verifying Against scikit-learn

We can verify our understanding by comparing our network's structure with scikit-learn's `MLPClassifier`, which implements the same forward propagation internally.

In [ ]:
# Compare our parameter count with sklearn MLP
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1, random_state=42)
mlp.fit(X_tr, y_tr)

sklearn_params = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
our_params = count_parameters([2, 16, 8, 1])

print(f"\nsklearn MLP params: {sklearn_params}")
print(f"Our network params: {our_params}")
print(f"Match: {sklearn_params == our_params}")

print(f"\nsklearn layer weight shapes: {[w.shape for w in mlp.coefs_]}")
print(f"Our layer weight shapes:     {[w.shape for w in nn.weights]}")

---

## 13. Summary

Forward propagation passes input through each layer: compute the weighted sum $\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$, apply the activation $\mathbf{a}^{[l]} = f^{[l]}(\mathbf{z}^{[l]})$, repeat until the output layer.

Key concepts: **matrix notation** for vectorized computation, **batch processing** for efficiency, **output activations** matched to the task (sigmoid, softmax, linear), **parameter counting**, and **weight initialization** (He/Xavier). The **computational graph** and **cache** of intermediate values set the stage for backpropagation.

Forward propagation answers: *given weights and input, what is the prediction?* Training answers the reverse: *given inputs and desired outputs, how should weights change?* That reverse pass — backpropagation — computes gradients and completes the learning loop.